In [ ]:
from modules.preprocessing import *
from modules.modeling import *
from modules.visualizing import *


In [2]:
input_path = "data/DrugBank/"
output_path = "output/"

col_smiles = "SMILES"
col_index = "INCHIKEY"

In [ ]:
input_file_name = "raw_drugbank_5.1.13.csv"
df = pd.read_csv(os.path.join(input_path, input_file_name))
df.rename(columns={"Common name": "PREFERRED_NAME",
                   "CAS": "CASRN",
                   "Standard InChI Key": "INCHIKEY",
                   }, inplace=True)
df_structures = df.dropna(subset=["INCHIKEY"]).reset_index(drop=True)


In [ ]:
import pubchempy as pcp
import time

def get_pubchem_data(df, col_inchikey, col_cas, col_name, output_file, resume=False):
    """ Fetches IUPAC name, SMILES, and InChI strings from PubChem for a list of chemicals 
    provided in dataframe with their InChIKeys, CAS numbers and chemical names, saving regularly to avoid data loss.
    Allows to resume from last saved compound.

    Inputs
    ----------
    df : pandas dataframe, mandatory
        Dataframe containing a list of chemicals with columns for CAS numbers and chemical names
    col_inchikey: str, mandatory
        column name containing InChI keys
    col_cas: str, mandatory
        column name containing CAS numbers
    col_name: str, mandatory
        column name containing chemical names
    output_file: str, mandatory
        path to output csv file
    resume: bool, optional, default=False
        if True and output_file exists, resume from last saved compound

    Outputs
    ----------
    df_out: pandas dataframe
        dataframe with CAS, chemical name, foundby (CAS or name), PubChem ID (CID), IUPAC name, 
        isomeric and caonical SMILES, InChI key and InChI strings
    """

    if resume & os.path.exists(output_file):
        df_out = pd.read_csv(output_file)
        done_set = set(df_out[col_inchikey])
        print(f"Resuming: {len(done_set)} compounds already processed.")
    else:
        df_out = pd.DataFrame(columns=[col_cas, col_name, col_inchikey, 'Found by', 'CID', 'IUPAC', 'InChI', 'SMILES'])
        done_set = set()


    for i, (cas, name, inchikey) in enumerate(zip(df[col_cas], df[col_name], df[col_inchikey])):

        if inchikey in done_set:  
            continue

        compound_data = None
        foundby = None

        try:
            results = pcp.get_compounds(inchikey, 'inchikey')
            if results:
                compound_data, foundby = results[0], 'InChIKey'
        except:
            pass

        if compound_data is None:
            try:
                results = pcp.get_compounds(cas, 'name')
                if results:
                    compound_data, foundby = results[0], 'CAS'
            except:
                pass

        if compound_data is None:
            try:
                results = pcp.get_compounds(name, 'name')
                if results:
                    compound_data, foundby = results[0], 'name'
            except:
                pass

        if compound_data:
            row = pd.DataFrame([[cas, name, inchikey, foundby,
                                compound_data.cid,
                                compound_data.iupac_name,
                                compound_data.inchi,
                                compound_data.smiles]],
                            columns=df_out.columns)
        else:
            row = pd.DataFrame([[cas, name, inchikey, "not found", None, None, None, None]],
                            columns=df_out.columns)

        df_out = pd.concat([df_out, row], ignore_index=True)

        if i % 50 == 0:
            df_out.to_csv(output_file, index=False)
            print(f"Progress saved at index {i} ({cas}, {name})")

        time.sleep(0.2)

    df_out.to_csv(output_file, index=False)

    return df_out

In [7]:
output_file = os.path.join(output_path, "drugbank_5.1.13_pubchem.csv")
df_pubchem = get_pubchem_data(df_structures, 'INCHIKEY', 'CASRN', 'PREFERRED_NAME', output_file, resume=False)
df_pubchem

Progress saved at index 0 (128270-60-0, Bivalirudin)
Progress saved at index 50 (58-68-4, NADH)
Progress saved at index 100 (55142-85-3, Ticlopidine)
Progress saved at index 150 (62-54-4, Calcium acetate)
Progress saved at index 200 (53643-48-4, Vindesine)
Progress saved at index 250 (62989-33-7, Sapropterin)
Progress saved at index 300 (968-81-0, Acetohexamide)
Progress saved at index 350 (74103-06-3, Ketorolac)
Progress saved at index 400 (54965-21-8, Albendazole)
Progress saved at index 450 (298-57-7, Cinnarizine)
Progress saved at index 500 (127-33-3, Demeclocycline)
Progress saved at index 550 (79350-37-1, Cefixime)
Progress saved at index 600 (390-28-3, Methoxamine)
Progress saved at index 650 (135-09-1, Hydroflumethiazide)
Progress saved at index 700 (28657-80-9, Cinoxacin)
Progress saved at index 750 (55-56-1, Chlorhexidine)
Progress saved at index 800 (59122-46-2, Misoprostol)
Progress saved at index 850 (57-47-6, Physostigmine)
Progress saved at index 900 (50-44-2, Mercaptopu

,CASRN,PREFERRED_NAME,INCHIKEY,Found by,CID,IUPAC,InChI,SMILES
0,128270-60-0,Bivalirudin,OIRCOABEOLEUMC-GEJPAHFPSA-N,InChIKey,16129704,(2S)-2-[[(2S)-2-[[(2S)-2-[[(2S)-2-[[(2S)-1-[(2...,InChI=1S/C98H138N24O33/c1-5-52(4)82(96(153)122...,CC[C@H](C)[C@@H](C(=O)N1CCC[C@H]1C(=O)N[C@@H](...
1,65807-02-5,Goserelin,BLCLNMBMMGCOAS-URPVMXJPSA-N,InChIKey,5311128,(2S)-N-[(2S)-1-[[(2S)-1-[[(2S)-1-[[(2S)-1-[[(2...,InChI=1S/C59H84N18O14/c1-31(2)22-40(49(82)68-3...,CC(C)C[C@@H](C(=O)N[C@@H](CCCN=C(N)N)C(=O)N1CC...
2,1405-97-6,Gramicidin D,NDAYQJDHGXTBJL-MWWSRJDJSA-N,InChIKey,45267103,(2R)-2-[[2-[[(2S)-2-formamido-3-methylbutanoyl...,InChI=1S/C96H135N19O16/c1-50(2)36-71(105-79(11...,C[C@@H](C(=O)N[C@H](C(C)C)C(=O)N[C@@H](C(C)C)C...
3,16679-58-6,Desmopressin,NFLWUMRGJYTJIN-PNIOQBSNSA-N,InChIKey,5311065,(2S)-N-[(2R)-1-[(2-amino-2-oxoethyl)amino]-5-(...,InChI=1S/C46H64N14O12S2/c47-35(62)15-14-29-40(...,C1C[C@H](N(C1)C(=O)[C@@H]2CSSCCC(=O)N[C@H](C(=...
4,120287-85-6,Cetrorelix,SBNPWPIBESPSIF-MHWMIDJBSA-N,InChIKey,16130924,None,InChI=1S/C70H92ClN17O14/c1-39(2)31-52(61(94)82...,C[C@H](C(=O)N)NC(=O)[C@@H]1CCCN1C(=O)[C@H](CCC...
...,...,...,...,...,...,...,...,...
12311,2567686-02-4,Tulmimetostat,CAAWBLRXQXMGHV-JAGNJQOISA-N,CAS,155590998,(2R)-7-chloro-2-[4-(3-methoxyazetidin-1-yl)cyc...,InChI=1S/C28H36ClN3O5S/c1-15-10-23(38-5)21(27(...,CC1=CC(=C(C(=O)N1)CNC(=O)C2=CC(=C3C(=C2C)O[C@@...
12312,2755812-39-4,Ibuzatrelvir,WGNWEPPRWQKSKI-AIEDFZFUSA-N,InChIKey,163362000,"methyl N-[(2S)-1-[(2S,4R)-2-[[(1S)-1-cyano-2-[...","InChI=1S/C21H30F3N5O5/c1-20(2,3)15(28-19(33)34...",CC(C)(C)[C@@H](C(=O)N1C[C@@H](C[C@H]1C(=O)N[C@...
12313,22393-86-8,Cetyl oleate,JYTMDBGMUIAIQH-ZPHPHTNESA-N,InChIKey,5377655,hexadecyl (Z)-octadec-9-enoate,InChI=1S/C34H66O2/c1-3-5-7-9-11-13-15-17-19-20...,CCCCCCCCCCCCCCCCOC(=O)CCCCCCC/C=C\CCCCCCCC
12314,64660-84-0,Cetyl myristoleate,DYIOQMKBBPSAFY-BENRWUELSA-N,InChIKey,6443825,hexadecyl (Z)-tetradec-9-enoate,InChI=1S/C30H58O2/c1-3-5-7-9-11-13-15-16-17-19...,CCCCCCCCCCCCCCCCOC(=O)CCCCCCC/C=C\CCCC


In [8]:
df_out = df_structures.merge(df_pubchem[["CID", "INCHIKEY", "InChI", "SMILES"]], on="INCHIKEY", how="left")
df_out.fillna({'SMILES': ''}, inplace=True)
df_out.to_csv(os.path.join(input_path, "input_drugbank_5.1.13.csv"), index=False)

In [ ]:
hyperparameters = dict(n_components=2, perplexity=100, n_iter=2000, learning_rate='auto', neighbors='pynndescent', initialization='pca', metric='jaccard', random_state=42,  n_jobs=-1, verbose=True)

from openTSNE.sklearn import TSNE

def fit_tsne_pipeline(df, col_smiles, col_index, hyperparameters):
    # -----------------------------
    # DATA PREPARATION
    # -----------------------------
    # Standardization
    df['standardized SMILES'] = standardize_smiles_df(df, col_smiles)

    # Fingerprint caculation
    fpsize = 1024
    fingerprints = calculate_descriptors_morgan_df(df, 'standardized SMILES', radius=2, fpSize=fpsize)
    df_fingerprints = pd.concat([df, fingerprints], axis=1)
    df_fingerprints.dropna(thresh=fpsize, inplace=True)
    fps = np.array(df_fingerprints.iloc[:, -1024:].astype('bool'))

    # -----------------------------
    # MODELLING
    # -----------------------------
    tsne = TSNE(**hyperparameters)
    df_tsne_fit = pd.DataFrame(tsne.fit_transform(fps), columns=['TSNE1', 'TSNE2'])
    df_tsne_fit.index = df_fingerprints[col_index]

    return df_tsne_fit, tsne

In [ ]:
df_tsne_fit, tsne = fit_tsne_pipeline(df_out, col_smiles, col_index, hyperparameters)
df_plot = df_out.merge(df_tsne_fit, on=col_index, how="left")

In [ ]:
fig_grey = px.scatter(df_plot, x="TSNE1", y="TSNE2", hover_name = 'PREFERRED_NAME', hover_data=['CASRN', 'standardized SMILES'],
                    render_mode="webgl", height=700, width=1200)

fig_grey.update_traces(
    marker=dict(color='black',size=3, opacity=0.3, line=dict(width=0)),
    name='grey_market')

fig_grey.update_layout(
    paper_bgcolor='white',
    plot_bgcolor='white',
    font_color='black',
    xaxis=dict(title="TSNE1",visible=False, showgrid=False, zeroline=False, 
                fixedrange=False),
    yaxis=dict(title="TSNE2",visible=False, showgrid=False, zeroline=False, 
                fixedrange=False))